# FaceSwap Studio - Kaggle T4 x2 Dual-GPU Launcher

This notebook launches **FaceSwap Studio** on Kaggle's dual NVIDIA T4 GPUs (`T4 x2`).
It initializes the runtime environment, prefetches FaceFusion v3 model weights, starts the asynchronous FastAPI backend server, and establishes a secure Cloudflare Quick Tunnel (`*.trycloudflare.com`) for connecting your browser studio UI.

### Step 1: Verify Hardware & Dual-GPU Visibility
Ensure the Kaggle session accelerator is configured with **GPU T4 x2**.

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

assert torch.cuda.device_count() >= 2, "Warning: Dual GPUs (T4 x2) not detected. Check Kaggle Session options."

### Step 2: Environment Setup & Model Weight Prefetching
Clones FaceFusion v3, installs required libraries (FFmpeg, FastAPI, ONNX Runtime GPU), downloads Cloudflared, and links models from `/kaggle/input/ff-models-cache` into `/kaggle/working/facefusion/.assets/models`.

In [ ]:
!bash scripts/setup_env.sh
!python3 scripts/prefetch_models.py

### Step 3: Launch FastAPI Background Server
Starts the non-blocking job manager server on port `7860`.

In [ ]:
import subprocess
import time
import requests

print("[Server] Starting FaceSwap Studio backend on port 7860...")
server_process = subprocess.Popen(
    ["python3", "-m", "backend.server"],
    stdout=open("/kaggle/working/server.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for server to become responsive
max_retries = 30
server_ready = False
for attempt in range(max_retries):
    try:
        res = requests.get("http://127.0.0.1:7860/api/system/status", timeout=2)
        if res.status_code == 200:
            print("[Server] Backend API is up and running!")
            print(res.json())
            server_ready = True
            break
    except Exception:
        time.sleep(1)

if not server_ready:
    print("[Error] Server failed to start. Showing last log output:")
    with open("/kaggle/working/server.log", "r") as f:
        print(f.read())
    raise RuntimeError("Backend server startup timed out.")

### Step 4: Launch Cloudflare Quick Tunnel
Exposes port `7860` to the internet via Cloudflare Quick Tunnel without requiring account credentials or port forwarding.

In [ ]:
import re
import time
from IPython.display import HTML, display

tunnel_bin = "/kaggle/working/cloudflared"
tunnel_log = "/kaggle/working/tunnel.log"

print("[Tunnel] Spawning Cloudflare Quick Tunnel...")
tunnel_process = subprocess.Popen(
    [tunnel_bin, "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

# Search log file for generated URL
for _ in range(40):
    time.sleep(1)
    try:
        with open(tunnel_log, "r") as f:
            content = f.read()
            match = url_pattern.search(content)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    card_html = f"""
    <div style="background: #0b0e14; border: 1px solid #00f0ff; border-radius: 12px; padding: 20px; color: #ffffff; font-family: sans-serif; box-shadow: 0 0 20px rgba(0, 240, 255, 0.2);">
      <h2 style="margin: 0 0 10px 0; color: #00f0ff;">FaceSwap Studio Remote Tunnel Ready</h2>
      <p style="margin: 0 0 14px 0; color: #8e9bb0;">Open the link below in your browser or paste it into the Remote Backend URL field in your local studio UI:</p>
      <div style="margin-bottom: 12px;">
        <a href="{public_url}" target="_blank" style="background: linear-gradient(135deg, #00c6ff, #0072ff); color: white; padding: 10px 20px; border-radius: 6px; text-decoration: none; font-weight: bold; display: inline-block;">
          Open FaceSwap Studio Web UI
        </a>
      </div>
      <div style="font-family: monospace; background: #040507; padding: 8px 12px; border-radius: 6px; border: 1px solid rgba(255,255,255,0.1); color: #00ff88; word-break: break-all;">
        {public_url}
      </div>
    </div>
    """
    display(HTML(card_html))
else:
    print("[Warning] Cloudflare URL not detected within 40 seconds. Check /kaggle/working/tunnel.log:")
    with open(tunnel_log, "r") as f:
        print(f.read())

### Step 5: Keep-Alive & Job Heartbeat Loop
Runs an active monitoring loop to display server status and keep the notebook session active.

In [ ]:
import time
import requests
from datetime import datetime

print("[Monitor] Keep-alive monitoring loop started. Press Stop in Kaggle to terminate.")
while True:
    try:
        res = requests.get("http://127.0.0.1:7860/api/jobs", timeout=5)
        if res.status_code == 200:
            jobs = res.json()
            active = [j for j in jobs if j.get("status") == "processing"]
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] Server Active | Total Jobs: {len(jobs)} | Processing: {len(active)}", end="\r")
    except Exception as e:
        print(f"\n[Monitor Error] {e}")
    time.sleep(10)